# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a complete guide for loading and exploring the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. This exploration will utilize the unique `@id` fields to reference the entity structure.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available record sets from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', None)}: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review the available record sets, fields, and their unique `@id` values.

Record sets and fields are core to the Croissant data model. We'll print all available record sets and their fields, referencing each entity by its `@id`.

In [ ]:
def print_recordsets_and_fields(ds):
    print("Available Record Sets:")
    record_sets = ds.metadata.record_sets
    for rs in record_sets:
        print(f"  - Name: {getattr(rs, 'name', 'N/A')} | @id: {rs.id}")
        print("    Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"      - {getattr(field, 'name', 'N/A')} (@id: {field.id}, column: {getattr(field, 'column_id', None)})")

print_recordsets_and_fields(dataset)

# Display one record from the first available record set
print("\nSample record from the first record set:")
if dataset.metadata.record_sets:
    first_rs_id = dataset.metadata.record_sets[0].id
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if i >= 0:  # Print only the first record
            break

## 3. Data Extraction
Load tabular data from the available record sets into pandas DataFrames for further analysis.

**Note:** All data extraction uses the `@id` fields for record sets and columns.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record Set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"--> Loaded DataFrame for Record Set: {record_set_id} (Shape: {df.shape})")

# Show columns from the first record set
first_record_set_id = record_set_ids[0]
print(f"\nColumns for Record Set {first_record_set_id}:\n", dataframes[first_record_set_id].columns.tolist())
dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform typical data processing and basic statistical analysis:
- Filter proteins by a numeric field (e.g., coverage percentage, molecular weight, or peptide_count).
- Normalize the numeric field.
- Optionally group data by another field (e.g., 'gene_name' or protein class).

**All fields are referenced by their Croissant `@id`. Please refer to the Data Overview for exact field identifiers.**

In [ ]:
# Example field @ids for the main record set
# Please update these IDs if the actual dataset changes

# Let's inspect the first record set for numeric fields:
print("Fields of the first record set and their @ids:")
main_rs = dataset.metadata.record_sets[0]
for field in main_rs.fields:
    print(f"Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")

# For demonstration, suppose the field IDs for peptide count and gene name are as follows:
numeric_field_id = None
group_field_id = None
for field in main_rs.fields:
    if any(x in field.name.lower() for x in ["peptide", "count"]):
        numeric_field_id = field.id
    if group_field_id is None and ("gene" in field.name.lower()):
        group_field_id = field.id

if not numeric_field_id:
    numeric_field_id = main_rs.fields[0].id  # fallback to any field

if not group_field_id:
    group_field_id = main_rs.fields[1].id if len(main_rs.fields) > 1 else main_rs.fields[0].id

df = dataframes[main_rs.id]
print(f"\nUsing numeric field: {numeric_field_id}")

# Remove records where the selected field is missing or not numeric
df_valid = df.copy()
df_valid = df_valid[pd.to_numeric(df_valid[numeric_field_id], errors='coerce').notnull()]
df_valid[numeric_field_id] = df_valid[numeric_field_id].astype(float)

# Filter: select records with numeric field > threshold
threshold = 10
filtered_df = df_valid[df_valid[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional: Group by gene name or another categorical field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id}, showing mean {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and the relationship between numeric value and group attribute (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group field is available and not too many unique values, plot boxplot
if group_field_id in filtered_df.columns and filtered_df[group_field_id].nunique() < 50:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.xticks(rotation=45)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and parsed a Croissant-structured biomedical dataset using `mlcroissant`.
- Explored available record sets, fields, and columns using their unique `@id`s for robust referencing.
- Loaded data into pandas for flexible processing.
- Filtered, normalized, grouped, and visualized key numeric attributes, showcasing typical data science workflows.

For your own analysis, substitute the field and record set `@id` values as needed, referring to the data overview cells. This workflow ensures reproducible, schema-aware, and FAIR-aligned data analytics!